In [16]:
import pandas as pd
import json
import requests
from bs4 import BeautifulSoup
import time
import re
from datetime import datetime
from collections import Counter
from tqdm import tqdm

In [63]:
def preparing_json(json_file):
    with open(json_file, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    vacancies = data
    
    def get_field_value(data, field_path):
        if isinstance(data, dict):
            value = data.get(field_path, '')
        else:
            value = ''
        
        if isinstance(value, dict):
            return value.get('name', '')
        elif isinstance(value, list):
            names = []
            for item in value:
                if isinstance(item, dict) and 'name' in item:
                    names.append(item['name'])
            return ', '.join(names)
        else:
            return value
    
    parsed_vacancies = []
    seen_ids = set()
    
    for v in vacancies:
        vac_id = v.get('id', '')

        if vac_id in seen_ids:
            continue
        seen_ids.add(vac_id)
        
        parsed = {
            'ID': vac_id,
            'Название': v.get('name', ''),
            'Департамент': get_field_value(v, 'department'),
            'Город': get_field_value(v, 'area'),
            'Опыт': get_field_value(v, 'experience'),
            'Расписание': get_field_value(v, 'schedule'),
            'Формат работы': get_field_value(v, 'work_format'),
            'Рабочие часы': get_field_value(v, 'working_hours'),
            'График': get_field_value(v, 'work_schedule_by_days'),
            'Зарплата': 'Не указана',
            'Дата публикации': v.get('published_at', ''),
            'Ссылка': v.get('alternate_url', '')
        }

        salary = v.get('salary')
        if salary and isinstance(salary, dict):
            salary_from = salary.get('from')
            salary_to = salary.get('to')
            if salary_from is not None and salary_to is not None:
                parsed['Зарплата'] = f"{salary_from:,} - {salary_to:,}".replace(',', ' ')
            elif salary_from is not None:
                parsed['Зарплата'] = f"от {salary_from:,}".replace(',', ' ')
            elif salary_to is not None:
                parsed['Зарплата'] = f"до {salary_to:,}".replace(',', ' ')
        
        parsed_vacancies.append(parsed)
    
    df = pd.DataFrame(parsed_vacancies)
    return df

df = preparing_json('vacancies.json')

In [64]:
df.head(2)

,ID,Название,Департамент,Город,Опыт,Расписание,Формат работы,Рабочие часы,График,Зарплата,Дата публикации,Ссылка
0,131993208,Младший менеджер по персоналу,Яндекс,Москва,От 1 года до 3 лет,Полный день,Гибрид,8 часов,5/2,Не указана,2026-04-09T14:49:42+0300,https://hh.ru/vacancy/131993208
1,131915650,Дизайнер мебели,Яндекс,Москва,Нет опыта,Полный день,На месте работодателя,6 часов,5/2,Не указана,2026-04-07T13:39:37+0300,https://hh.ru/vacancy/131915650


In [ ]:
def extract_russian_text(html_content):
    if not html_content:
        return ""
    
    soup = BeautifulSoup(html_content, 'html.parser')
    for tag in soup(['script', 'style', 'noscript', 'meta', 'link', 'header', 'footer', 'nav']):
        tag.decompose()
    text = soup.get_text(separator='\n')
    lines = []
    for line in text.split('\n'):
        line = line.strip()
        if line and len(line) > 30: 
            if re.search('[а-яА-Я]', line):
                skip_patterns = [
                    'cookie', 'Cookie', 'файлы cookie', 'файлов cookie',
                    'Произошла ошибка', 'Попробуйте перезагрузить',
                    'Соискателям', 'Работодателям',
                    'Создать резюме', 'Готовое резюме', 'Карьерная консультация',
                    'Все сервисы', 'Подпишитесь на push-уведомления',
                    'Подписаться', 'push-уведомления',
                    '© 2026', 'Хэдхантер', 'Реклама на сайте', 'Требования к ПО',
                    'Безопасный HeadHunter', 'Каталог компаний',
                    'Работа рядом с метро', 'Сетка: соцсеть для нетворкинга',
                    'Профориентация', 'Социальная поддержка при сложных жизненных ситуациях',
                    'Производственный календарь', 'Экспертная рекомендация',
                    'Рынок труда', 'Жизнь в компании',
                    'Рейтинг работодателей России', 'Боты для уведомлений',
                    'Мобильное приложение', 'Этика и комплаенс',
                    'Оказание услуг', 'Использование сайтов',
                    'Защита персональных данных', 'Пользовательское соглашение',
                    'рекомендательные технологии', 'информационном ресурсе', 'работодатель мог связаться с вами', 'полностью принимаете условия', 
                    'Соглашения об оказании услуг по содействию в трудоустройстве (оферта)', ' (информационные технологии предоставления информации на основе сбора, систематизации и анализа сведений, относящихся к предпочтениям пользователей сети «Интернет», находящихся на территории Российской Федерации)',
                    'Похожие вакансии в этой компании', 'Карьера для молодых специалистов', 'Карьера в некоммерческих организациях',
                    'Показывает отзывы от сотрудников', 'Соглашения об оказании услуг по содействию в трудоустройстве (оферта)'
                ]
                
                skip = False
                for pattern in skip_patterns:
                    if pattern.lower() in line.lower():
                        skip = True
                        break
                
                if not skip:
                    lines.append(line)

    result = '\n'.join(lines)
    result = re.sub(r'\n{3,}', '\n\n', result)
    
    return result

def fetch_vacancy_html(alternate_url, headers):
    try:
        response = requests.get(alternate_url, headers=headers, timeout=10)
        response.raise_for_status()
        return response.text
    except Exception as e:
        print(f"Ошибка загрузки {alternate_url}: {e}")
        return None

with open('vacancies.json', 'r', encoding='utf-8') as f:
    vacancies = json.load(f)

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

all_vacancies = []
seen_ids = set()

for idx, vacancy in enumerate(tqdm(vacancies, desc="Обработка вакансий", unit="вакансия")):
    vac_id = vacancy.get('id')
    if vac_id in seen_ids:
        continue
    seen_ids.add(vac_id)
    
    alternate_url = vacancy.get('alternate_url')
    if not alternate_url:
        continue
    html_content = fetch_vacancy_html(alternate_url, headers)
    russian_text = extract_russian_text(html_content) if html_content else ""
    
    result = {
        'id': vac_id,
        'name': vacancy.get('name'),
        'company_id': vacancy.get('company_id'),
        'alternate_url': alternate_url,
        'text': russian_text,
        'fetched_at': datetime.now().isoformat()
    }
    
    all_vacancies.append(result)
    time.sleep(1)

In [66]:
df_clean_all = pd.DataFrame(all_vacancies)
df_clean = df_clean_all[df_clean_all['text'] != '']

In [67]:
len(df_clean)

460

In [ ]:
all_lines = []
for text in df_clean['text'].astype(str):
    all_lines.extend(text.split('\n'))

frequency = Counter(all_lines)

total_vacancies = len(df_clean)
threshold = total_vacancies * 0.3

phrase_in_vacancies = {}

for idx, text in enumerate(df_clean['text'].astype(str)):
    lines = set(text.split('\n'))
    for line in lines:
        if line.strip():
            phrase_in_vacancies[line] = phrase_in_vacancies.get(line, 0) + 1

common_phrases = [phrase for phrase, count in phrase_in_vacancies.items() 
                  if count >= threshold]

print(f"Найдено {len(common_phrases)} фраз, встречающихся в {threshold:.0f}+ вакансиях:")
for phrase in common_phrases[:10]:
    print(f"  - '{phrase}'")

def remove_common_phrases(text):
    if pd.isna(text):
        return text
    lines = text.split('\n')
    filtered_lines = [line for line in lines if line not in common_phrases]
    return '\n'.join(filtered_lines)

df_clean['text_cleaned'] = df_clean['text'].apply(remove_common_phrases)

In [69]:
df['Текст'] = df_clean['text_cleaned']
df['ID компании'] = df_clean['company_id']

In [70]:
df.head(2)

,ID,Название,Департамент,Город,Опыт,Расписание,Формат работы,Рабочие часы,График,Зарплата,Дата публикации,Ссылка,Текст,ID компании
0,131993208,Младший менеджер по персоналу,Яндекс,Москва,От 1 года до 3 лет,Полный день,Гибрид,8 часов,5/2,Не указана,2026-04-09T14:49:42+0300,https://hh.ru/vacancy/131993208,Вакансия Младший менеджер по персоналу в Москв...,1740.0
1,131915650,Дизайнер мебели,Яндекс,Москва,Нет опыта,Полный день,На месте работодателя,6 часов,5/2,Не указана,2026-04-07T13:39:37+0300,https://hh.ru/vacancy/131915650,"Вакансия Дизайнер мебели в Москве, работа в ко...",1740.0


In [71]:
result = df[df['Текст'].notna()]
len(result)

460

In [73]:
result.to_csv("vacancies.csv")